# 01 — Setup: NorthwindDW no Spark + Delta Lake (Spark SQL)

Cria SparkSession, databases e tabelas Delta via `spark.sql()` DDL.

**Versão:** Spark SQL — toda interação com o catálogo via SQL DDL.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, WAREHOUSE_DIR

spark = get_spark("NorthwindDW SQL - 01 Setup")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:41:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
for db in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
    print(f"Database '{db}' OK")

Database 'bronze' OK
Database 'silver' OK
Database 'gold' OK


In [3]:
# Bronze DDL
bronze_ddl = {
    "bronze.customers": (
        "CustomerID STRING, CompanyName STRING, ContactName STRING,"
        " ContactTitle STRING, Address STRING, City STRING, Region STRING,"
        " PostalCode STRING, Country STRING, Phone STRING, Fax STRING,"
        " _LoadTimestamp TIMESTAMP"
    ),
    "bronze.employees": (
        "EmployeeID INT, LastName STRING, FirstName STRING, Title STRING,"
        " TitleOfCourtesy STRING, BirthDate TIMESTAMP, HireDate TIMESTAMP,"
        " Address STRING, City STRING, Region STRING, PostalCode STRING,"
        " Country STRING, HomePhone STRING, Extension STRING,"
        " ReportsTo INT, PhotoPath STRING, _LoadTimestamp TIMESTAMP"
    ),
    "bronze.products": (
        "ProductID INT, ProductName STRING, SupplierID INT, CategoryID INT,"
        " QuantityPerUnit STRING, UnitPrice DOUBLE, UnitsInStock INT,"
        " UnitsOnOrder INT, ReorderLevel INT, Discontinued BOOLEAN,"
        " _LoadTimestamp TIMESTAMP"
    ),
    "bronze.categories": (
        "CategoryID INT, CategoryName STRING, Description STRING,"
        " _LoadTimestamp TIMESTAMP"
    ),
    "bronze.suppliers": (
        "SupplierID INT, CompanyName STRING, ContactName STRING,"
        " ContactTitle STRING, Address STRING, City STRING, Region STRING,"
        " PostalCode STRING, Country STRING, Phone STRING, Fax STRING,"
        " _LoadTimestamp TIMESTAMP"
    ),
    "bronze.shippers": (
        "ShipperID INT, CompanyName STRING, Phone STRING, _LoadTimestamp TIMESTAMP"
    ),
    "bronze.orders": (
        "OrderID INT, CustomerID STRING, EmployeeID INT,"
        " OrderDate TIMESTAMP, RequiredDate TIMESTAMP, ShippedDate TIMESTAMP,"
        " ShipVia INT, Freight DOUBLE, ShipName STRING, ShipAddress STRING,"
        " ShipCity STRING, ShipRegion STRING, ShipPostalCode STRING,"
        " ShipCountry STRING, _LoadTimestamp TIMESTAMP"
    ),
    "bronze.order_details": (
        "OrderID INT, ProductID INT, UnitPrice DOUBLE,"
        " Quantity INT, Discount DOUBLE, _LoadTimestamp TIMESTAMP"
    ),
    "bronze.territories": (
        "TerritoryID STRING, TerritoryDescription STRING,"
        " RegionID INT, _LoadTimestamp TIMESTAMP"
    ),
    "bronze.region": "RegionID INT, RegionDescription STRING, _LoadTimestamp TIMESTAMP",
    "bronze.employee_territories": (
        "EmployeeID INT, TerritoryID STRING, _LoadTimestamp TIMESTAMP"
    ),
}

for table, cols in bronze_ddl.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} ({cols}) USING DELTA")
    print(f"  {table} OK")
print(f"\nBronze: {len(bronze_ddl)} tabelas criadas.")

  bronze.customers OK


  bronze.employees OK


  bronze.products OK


  bronze.categories OK


  bronze.suppliers OK


...

  bronze.orders OK


  bronze.order_details OK


  bronze.territories OK


  bronze.region OK


  bronze.employee_territories OK

Bronze: 11 tabelas criadas.


In [4]:
# Silver DDL
silver_ddl = {
    "gold.DimCustomer": (
        "CustomerSK INT, CustomerID STRING, CompanyName STRING,"
        " ContactName STRING, ContactTitle STRING, City STRING, Country STRING,"
        " ValidFrom DATE, ValidTo DATE, IsCurrent BOOLEAN"
    ),
    "gold.DimEmployee": (
        "EmployeeSK INT, EmployeeID INT, FullName STRING, Title STRING,"
        " HireDate DATE, City STRING, Country STRING, ReportsToID INT,"
        " ManagerName STRING, TerritoryList STRING, RegionName STRING,"
        " LoadTimestamp TIMESTAMP"
    ),
    "gold.DimProduct": (
        "ProductSK INT, ProductID INT, ProductName STRING,"
        " CategoryName STRING, SupplierCompany STRING, UnitPrice DOUBLE,"
        " QuantityPerUnit STRING, Discontinued BOOLEAN,"
        " ValidFrom DATE, ValidTo DATE, IsCurrent BOOLEAN"
    ),
    "gold.DimCategory": (
        "CategorySK INT, CategoryID INT, CategoryName STRING,"
        " Description STRING, LoadTimestamp TIMESTAMP"
    ),
    "gold.DimSupplier": (
        "SupplierSK INT, SupplierID INT, CompanyName STRING,"
        " City STRING, Country STRING, LoadTimestamp TIMESTAMP"
    ),
    "gold.DimShipper": (
        "ShipperSK INT, ShipperID INT, CompanyName STRING,"
        " Phone STRING, LoadTimestamp TIMESTAMP"
    ),
    "gold.DimTerritory": (
        "TerritorySK INT, TerritoryID STRING, TerritoryDescription STRING,"
        " RegionID INT, RegionName STRING, LoadTimestamp TIMESTAMP"
    ),
    "gold.DimDate": (
        "DateKey INT, FullDate DATE, Year INT, Quarter INT, Month INT,"
        " MonthName STRING, Day INT, DayOfWeek INT, DayName STRING, IsWeekend BOOLEAN"
    ),
}

for table, cols in silver_ddl.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} ({cols}) USING DELTA")
    print(f"  {table} OK")
print(f"\nSilver: {len(silver_ddl)} tabelas criadas.")

  gold.DimCustomer OK


  gold.DimEmployee OK


  gold.DimProduct OK


  gold.DimCategory OK


  gold.DimSupplier OK


  gold.DimShipper OK


  gold.DimTerritory OK


  gold.DimDate OK

Silver: 8 tabelas criadas.


In [5]:
# Gold DDL
gold_ddl = {
    "gold.FactSales": (
        "SalesSK INT, OrderDateKey INT, CustomerSK INT, ProductSK INT,"
        " EmployeeSK INT, ShipperSK INT, OrderID INT, ProductID INT,"
        " UnitPrice DOUBLE, Quantity INT, Discount DOUBLE,"
        " GrossRevenue DOUBLE, NetRevenue DOUBLE, LoadTimestamp TIMESTAMP"
    ),
    "gold.FactOrderFulfillment": (
        "FulfillmentSK INT, OrderID INT, CustomerSK INT, EmployeeSK INT,"
        " ShipperSK INT, OrderDateKey INT, RequiredDateKey INT,"
        " ShippedDateKey INT, Freight DOUBLE, ShipCountry STRING,"
        " DaysToShip INT, IsLate BOOLEAN, LoadTimestamp TIMESTAMP"
    ),
    "gold.FactProductStock": (
        "StockSK INT, SnapshotDateKey INT, ProductSK INT, CategorySK INT,"
        " UnitsInStock INT, UnitsOnOrder INT, ReorderLevel INT,"
        " NeedsReorder BOOLEAN, LoadTimestamp TIMESTAMP"
    ),
}

for table, cols in gold_ddl.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} ({cols}) USING DELTA")
    print(f"  {table} OK")
print(f"\nGold: {len(gold_ddl)} tabelas criadas.")

  gold.FactSales OK


  gold.FactOrderFulfillment OK


  gold.FactProductStock OK

Gold: 3 tabelas criadas.


In [6]:
print("=" * 50)
print("Setup NorthwindDW — Spark SQL")
print("=" * 50)
for db in ["bronze", "silver", "gold"]:
    tables = spark.catalog.listTables(db)
    print(f"\n{db} ({len(tables)} tabelas):")
    for t in tables:
        print(f"  - {t.name}")
print("\nWarehouse:", WAREHOUSE_DIR)

Setup NorthwindDW — Spark SQL



bronze (11 tabelas):
  - categories
  - customers
  - employee_territories
  - employees
  - order_details
  - orders
  - products
  - region
  - shippers
  - suppliers
  - territories

silver (0 tabelas):



gold (11 tabelas):
  - dimcategory
  - dimcustomer
  - dimdate
  - dimemployee
  - dimproduct
  - dimshipper
  - dimsupplier
  - dimterritory
  - factorderfulfillment
  - factproductstock
  - factsales

Warehouse: /workspace/pf_northwind/spark_sql/warehouse
